# RMSE bar charts by company and regime

Data source: the three company summary tables in `ALL_TABLES.ipynb` (SPY, AAPL, MSFT) — regimes × models × rolling methods (none / monthly / daily).

One PDF, twelve charts: SPY (4 regimes), then AAPL (4), then MSFT (4). Each chart: models on the x-axis; three colored bars per model for the rolling methods.


In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.backends.backend_pdf import PdfPages

HERE = Path(".").resolve()
ALL_TABLES = HERE / "ALL_TABLES.ipynb"

COMPANIES = ["SPY", "AAPL", "MSFT"]
REGIMES = ["2008-2009", "2013-2014", "2018-2019", "2019-2020"]
MODELS = ["GBM", "Merton", "Heston–Merton", "GARCH–Merton"]
ROLLINGS = ["none", "monthly", "daily"]
COLORS = {"none": "#4C72B0", "monthly": "#55A868", "daily": "#C44E52"}


def parse_company_tables(nb_path: Path) -> dict[str, list[dict]]:
    """Read only the three big company tables from ALL_TABLES.ipynb."""
    nb = json.loads(nb_path.read_text(encoding="utf-8"))
    out: dict[str, list[dict]] = {}
    for cell in nb["cells"]:
        if cell.get("cell_type") != "markdown":
            continue
        src = "".join(cell.get("source", []))
        m = re.match(
            r"###\s+(SPY|AAPL|MSFT)\s+[—\-]\s+RMSE index",
            src.strip(),
        )
        if not m:
            continue
        company = m.group(1)
        rows = []
        for line in src.splitlines():
            line = line.strip()
            if not line.startswith("|") or line.startswith("|---") or "Regime" in line:
                continue
            parts = [p.strip() for p in line.strip("|").split("|")]
            if len(parts) < 5:
                continue
            rows.append(
                {
                    "Regime": parts[0],
                    "Model": parts[1],
                    "none": float(parts[2]),
                    "monthly": float(parts[3]),
                    "daily": float(parts[4]),
                }
            )
        out[company] = rows
    missing = [c for c in COMPANIES if c not in out]
    if missing:
        raise RuntimeError(f"Missing company tables in ALL_TABLES.ipynb: {missing}")
    return out


DATA = parse_company_tables(ALL_TABLES)
for c in COMPANIES:
    print(f"{c}: {len(DATA[c])} rows")


In [ ]:
def plot_regime_bars(company: str, regime: str, ax) -> None:
    rows = [r for r in DATA[company] if r["Regime"] == regime]
    by_model = {r["Model"]: r for r in rows}
    x = np.arange(len(MODELS))
    width = 0.25
    for i, rolling in enumerate(ROLLINGS):
        vals = [by_model[m][rolling] for m in MODELS]
        ax.bar(
            x + (i - 1) * width,
            vals,
            width,
            label=rolling,
            color=COLORS[rolling],
        )
    ax.set_xticks(x)
    ax.set_xticklabels(MODELS)
    ax.set_ylabel("RMSE")
    ax.set_xlabel("Model")
    ax.set_title(f"{company} — {regime}")
    ax.legend(frameon=False, title="rolling")


for company in COMPANIES:
    for regime in REGIMES:
        fig, ax = plt.subplots(figsize=(8.5, 4.2))
        plot_regime_bars(company, regime, ax)
        fig.tight_layout()
        plt.show()


In [ ]:
pdf_path = HERE / "RMSE_BAR_CHARTS.pdf"

with PdfPages(pdf_path) as pdf:
    for company in COMPANIES:
        for regime in REGIMES:
            fig, ax = plt.subplots(figsize=(8.5, 4.2))
            plot_regime_bars(company, regime, ax)
            fig.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)

print(f"Wrote {pdf_path}")
